In [51]:
!pip install --upgrade --quiet langchain langchain-neo4j neo4j langchain-groq


In [52]:
!pip install neo4j langchain-community

In [53]:
!pip install --no-deps neo4j langchain-community
!pip install --no-deps langchain-neo4j
!pip install --no-deps langchain langchain-core langchain-groq
!pip install --no-deps langchain langchain-core

In [ ]:
import os
os.environ["NEO4J_URI"] = " "
os.environ["NEO4J_USERNAME"] = " "
os.environ["NEO4J_PASSWORD"] = " "
os.environ["NEO4J_DATABASE"] = " " 
os.environ["GROQ_API_KEY"] = " "


In [55]:
from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    database=os.environ["NEO4J_DATABASE"]
)

# Verify connection
try:
    graph.query("RETURN 1 as test")
    print("Neo4j connection successful!")
except Exception as e:
    print(f"Neo4j connection error: {e}")


Neo4j connection successful!


In [57]:
!pip install langchain-experimental


In [61]:
!pip install -U langchain-core langchain-groq

  Using cached langchain_core-1.0.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached langchain_groq-1.0.0-py3-none-any.whl.metadata (1.7 kB)
Using cached langchain_core-1.0.1-py3-none-any.whl (467 kB)
Using cached langchain_groq-1.0.0-py3-none-any.whl (16 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
  Attempting uninstall: langchain-groq
    Found existing installation: langchain-groq 0.3.8
    Uninstalling langchain-groq-0.3.8:
      Successfully uninstalled langchain-groq-0.3.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.0.1 which is incompatible.
langchain-experimental 0.3.4 requires langchain-core<0.4.0,>=0.3.28, but you ha

In [62]:
!pip install langchain langchain-core langchain-groq

  Using cached langchain_core-0.3.79-py3-none-any.whl.metadata (3.2 kB)
INFO: pip is looking at multiple versions of langchain-groq to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_groq-0.3.8-py3-none-any.whl.metadata (2.6 kB)
Using cached langchain_core-0.3.79-py3-none-any.whl (449 kB)
Using cached langchain_groq-0.3.8-py3-none-any.whl (16 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.0.1
    Uninstalling langchain-core-1.0.1:
      Successfully uninstalled langchain-core-1.0.1
  Attempting uninstall: langchain-groq
    Found existing installation: langchain-groq 1.0.0
    Uninstalling langchain-groq-1.0.0:
      Successfully uninstalled langchain-groq-1.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-neo4j 0.6.0 requires langchain-co

In [63]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
llm_transformer = LLMGraphTransformer(llm=llm)


# Generation of Main Academic Knowledge Graph

In [64]:
from langchain_core.documents import Document

with open("/kaggle/input/updated-main-knowledge-info/Main_Knowledge_updated.txt", "r") as file:
    text = file.read()

documents = [Document(page_content=text)]
graph_documents = llm_transformer.convert_to_graph_documents(documents)
print(f"Nodes:{graph_documents[0].nodes}")
print(f"Relationships:{graph_documents[0].relationships}")

Nodes:[Node(id='Linear Algebra', type='Mathematics', properties={}), Node(id='Calculus', type='Mathematics', properties={}), Node(id='Probability', type='Mathematics', properties={}), Node(id='Statistics', type='Mathematics', properties={}), Node(id='Discrete Mathematics', type='Mathematics', properties={}), Node(id='Graph Theory', type='Mathematics', properties={}), Node(id='Number Theory', type='Mathematics', properties={}), Node(id='Combinatorics', type='Mathematics', properties={}), Node(id='Programming Languages', type='Computer science', properties={}), Node(id='Object-Oriented Programming', type='Programming paradigm', properties={}), Node(id='Functional Programming', type='Programming paradigm', properties={}), Node(id='Procedural Programming', type='Programming paradigm', properties={}), Node(id='Java', type='Programming language', properties={}), Node(id='Python', type='Programming language', properties={}), Node(id='C++', type='Programming language', properties={}), Node(id=

In [65]:
from langchain_core.documents import Document

In [66]:
from langchain_community.graphs import Neo4jGraph
import os
graph.add_graph_documents(graph_documents)
print("Graph data has been stored in Neo4j.")

Graph data has been stored in Neo4j.


## **Entity Extraction**

In [70]:
from langchain_groq import ChatGroq
from langchain_core.utils.json import parse_json_markdown
from typing import Dict, List

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
)

def create_system_prompt() -> str:
    return """You are a skill extraction specialist. Extract ONLY specific named skills, tools, technologies, and knowledge areas. 
    - Extract single-word or short-phrase skills ONLY
    - DO NOT include full sentences or descriptions
    - DO NOT include experience levels or qualifications
    - Each skill should be a clear, standalone term"""

def create_skill_extraction_prompt(text: str) -> str:
    return f"""Extract specific skills from the text below into these categories:

        Technical Skills: Specific tools, programming languages, software, platforms
        Example: Python, SQL, Tableau, Excel

        Domain Knowledge: Specific fields, subjects, methodologies
        Example: Machine Learning, Statistics, Data Mining

        Text: {text}

        Return ONLY in this JSON format:
        {{
            "technical_skills": ["skill1", "skill2"],
            "domain_knowledge": ["domain1", "domain2"]
        }}"""

def extract_skills(text: str) -> Dict[str, List[str]]:
    messages = [
        ("system", create_system_prompt()),
        ("human", create_skill_extraction_prompt(text))
    ]
    
    response = llm.invoke(messages)
    
    try:
        # Use parse_json_markdown instead of JsonOutputParser
        parsed_skills = parse_json_markdown(response.content)
        return parsed_skills
    except Exception as e:
        print(f"Error parsing response: {e}")
        return {
            "technical_skills": [],
            "domain_knowledge": []
        }


### Job Description Skills Extraction

In [71]:
# Job Description
text = """Sample data analyst job description
At [Company X], we’re proud to stand at the forefront of the Big Data revolution. Using the latest analytics tools and processes, we’re able to maximize our offerings and deliver unparalleled service and support. To help carry us even further, we’re searching for an experienced data analyst to join our team. The ideal candidate will be highly skilled in all aspects of data analytics, including mining, generation, and visualization. Additionally, this person should be committed to transforming data into readable, goal-oriented reports that drive innovation and growth.

Objectives of this role
Develop, implement, and maintain leading-edge analytics systems, taking complicated problems and building simple frameworks
Identify trends and opportunities for growth through analysis of complex datasets
Evaluate organizational methods and provide source-to-target mappings and information-model specification documents for datasets
Create best-practice reports based on data mining, analysis, and visualization
Evaluate internal systems for efficiency, problems, and inaccuracies, and develop and maintain protocols for handling, processing, and cleaning data
Work directly with managers and users to gather requirements, provide status updates, and build relationships
Responsibilities
Work closely with project managers to understand and maintain focus on their analytics needs, including critical metrics and KPIs, and deliver actionable insights to relevant decision-makers
Proactively analyze data to answer key questions for stakeholders or yourself, with an eye on what drives business performance, and investigate and communicate which areas need improvement in efficiency and productivity
Create and maintain rich interactive visualizations through data interpretation and analysis, with reporting components from multiple data sources
Define and implement data acquisition and integration logic, selecting an appropriate combination of methods and tools within the defined technology stack to ensure optimal scalability and performance of the solution
Develop and maintain databases by acquiring data from primary and secondary sources, and build scripts that will make our data evaluation process more flexible or scalable across datasets
Required skills and qualifications
Three or more years of experience mining data as a data analyst
Proven analytics skills, including mining, evaluation, and visualization
Technical writing experience in relevant areas, including queries, reports, and presentations
Strong SQL or Excel skills, with aptitude for learning other analytics tools
Preferred skills and qualifications
Bachelor’s degree (or equivalent) in mathematics, computer science, economics, or statistics
Experience with database and model design and segmentation techniques
Strong programming experience with frameworks, including XML, JavaScript, and ETL
Practical experience in statistical analysis through the use of statistical packages, including Excel, SPSS, and SAS
Proven success in a collaborative, team-oriented environment
"""

JD_skills = extract_skills(text)
print(JD_skills)

{'technical_skills': ['SQL', 'Excel', 'XML', 'JavaScript', 'ETL', 'SPSS', 'SAS'], 'domain_knowledge': ['Data Mining', 'Statistics', 'Data Analytics', 'Machine Learning']}


In [72]:
def create_relationship_prompt(new_skills: Dict[str, List[str]], existing_knowledge: Dict[str, List[str]]) -> str:
    return f"""Given:

EXISTING KNOWLEDGE GRAPH:
Technical Skills: {', '.join(existing_knowledge)}


NEW SKILLS TO INTEGRATE:
Technical Skills: {', '.join(new_skills['technical_skills'])}
Domain Knowledge: {', '.join(new_skills['domain_knowledge'])}

Generate ONLY accurate and well-established relationships between skills following these rules:
1. Only include relationships that are widely accepted in academia/industry
2. Focus on direct, clear hierarchical relationships
3. For tools, only state their primary purpose
4. Avoid broad or questionable generalizations
5. Each skill should only be related to its most direct parent/application

Return relationships in numbered list format.
Only include relationships you are completely confident about.Do not include any other text, explanations, or notes."""

In [73]:
def get_all_node_names():
    query = "MATCH (n) RETURN n"
    nodes = graph.query(query)
    for node in nodes:
        print(node)
    return [node['n']['id'] for node in nodes]

node_names = get_all_node_names()
print("Node Names:")
for name in node_names:
    print(name)


{'n': {'id': 'Api Design'}}
{'n': {'id': 'System Design'}}
{'n': {'id': 'Cloud Architecture'}}
{'n': {'id': 'Mathematics'}}
{'n': {'id': 'Computer Science'}}
{'n': {'id': 'Programming Paradigm'}}
{'n': {'id': 'Programming Language'}}
{'n': {'id': 'Programming Concept'}}
{'n': {'id': 'Database Language'}}
{'n': {'id': 'Database Type'}}
{'n': {'id': 'Database Management System'}}
{'n': {'id': 'Database Design'}}
{'n': {'id': 'Database Systems'}}
{'n': {'id': 'Artificial Intelligence'}}
{'n': {'id': 'Machine Learning'}}
{'n': {'id': 'Deep Learning'}}
{'n': {'id': 'Data Manipulation'}}
{'n': {'id': 'Numerical Computing'}}
{'n': {'id': 'Statistical Visualization'}}
{'n': {'id': 'Nlp'}}
{'n': {'id': 'Sentence Embedding'}}
{'n': {'id': 'Similarity Search'}}
{'n': {'id': 'Parameter-Efficient Fine-Tuning'}}
{'n': {'id': 'Efficient Llm Training'}}
{'n': {'id': 'Supervised Fine-Tuning'}}
{'n': {'id': 'Optimization'}}
{'n': {'id': 'Computer Systems'}}
{'n': {'id': 'Security'}}
{'n': {'id': 'Field'

In [74]:
from langchain_groq import ChatGroq
from typing import Dict, List

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.9,
)

existing_knowledge = node_names

new_skills = JD_skills
relationships = llm.invoke([
    ("system", "You are a knowledge graph relationship specialist. Create clear, ONLY really close relationships between technical concepts."),
    ("human", create_relationship_prompt(new_skills, existing_knowledge))
])

meaning=relationships.content


llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
llm_transformer = LLMGraphTransformer(llm=llm)
text = meaning
documents = [Document(page_content=text)]
graph_documents = llm_transformer.convert_to_graph_documents(documents)
print(f"Nodes:{graph_documents[0].nodes}")
print(f"Relationships:{graph_documents[0].relationships}")

graph = Neo4jGraph(refresh_schema=False)
graph.add_graph_documents(graph_documents)

Nodes:[Node(id='Sql', type='Database language', properties={}), Node(id='Excel', type='Data analytics', properties={}), Node(id='Xml', type='Database language', properties={}), Node(id='Javascript', type='Programming language', properties={}), Node(id='Etl', type='Data mining', properties={}), Node(id='Spss', type='Statistics', properties={}), Node(id='Sas', type='Statistics', properties={}), Node(id='Data Mining', type='Data analytics', properties={}), Node(id='Statistics', type='Data analytics', properties={}), Node(id='Data Analytics', type='Machine learning', properties={})]
Relationships:[Relationship(source=Node(id='Sql', type='Database language', properties={}), target=Node(id='Database Language', type='Database language', properties={}), type='IS_A', properties={}), Relationship(source=Node(id='Excel', type='Data analytics', properties={}), target=Node(id='Data Analytics', type='Data analytics', properties={}), type='IS_A', properties={}), Relationship(source=Node(id='Xml', typ

### Resume Skills Extraction

In [75]:
!pip install PyMuPDF python-docx

In [76]:
import fitz
import docx
from pathlib import Path
from typing import Union

class ResumeParser:
    def __init__(self, file_path: Union[str, Path]):
        self.file_path = Path(file_path)
        self.text = ""

    def extract_text(self) -> str:
        """Extract text based on file extension"""
        suffix = self.file_path.suffix.lower()
        
        try:
            if suffix == '.pdf':
                return self._extract_from_pdf()
            elif suffix == '.docx':
                return self._extract_from_docx()
            elif suffix == '.txt':
                return self._extract_from_txt()
            else:
                raise ValueError(f"Unsupported file format: {suffix}")
        except Exception as e:
            print(f"Error extracting text from {self.file_path}: {str(e)}")
            return ""

    def _extract_from_pdf(self) -> str:
        """Extract text from PDF using PyMuPDF"""
        text = []
        with fitz.open(self.file_path) as doc:
            for page in doc:
                text.append(page.get_text())
        return "\n".join(text)

    def _extract_from_docx(self) -> str:
        """Extract text from DOCX"""
        doc = docx.Document(self.file_path)
        return "\n".join([paragraph.text for paragraph in doc.paragraphs])

    def _extract_from_txt(self) -> str:
        """Extract text from TXT"""
        return self.file_path.read_text(encoding='utf-8')

In [77]:
def extract_resume_text(file_path: str) -> str:
    parser = ResumeParser(file_path)
    return parser.extract_text()

    
resume_file = "/kaggle/input/resumes/Hemanth-(RESUME).pdf"
text = extract_resume_text(resume_file)
print(text)

Hemanth Raju
hemanthraju@gmail.com — +91 7671853020 — LinkedIn — GitHub
Education
Vellore Institute of Technology (VIT), Chennai
Sept 2022 – Present
B.Tech in Computer Science Engineering (AI & ML Specialization)
CGPA: 8.80*/10
Narayana Junior College, Hyderabad
2020 – 2022
MPC, Class XII
Aggregate: 96.6% (966/1000)
Projects
Similar Document Template Matching Algorithm
Nov 2023
Python, OpenCV, OCR
• Developed a template extraction, comparison, and fraud detection model for medical document verification,
addressing complexities in template extraction, comparison, fraud detection.
• Region-of-Interest (ROI) Methods utilizes sophisticated techniques for precise extraction.
• Structural Similarity Index (SSIM) quantifies structural similarity to identify potential matches.
• Optical Character Recognition (OCR) extracts critical information such as patient details, provider information,
and billing amounts.
• Paper: arXiv
LLM and Knowledge Graph Based Interviewer Selection System
NOv 2024
P

In [78]:
resume_skills = extract_skills(text)
print(resume_skills)

{'technical_skills': ['Python', 'SQL', 'Java', 'C/C++', 'HTML/CSS', 'JavaScript', 'TensorFlow', 'Keras', 'Scikit-learn', 'Pandas', 'NumPy', 'Matplotlib', 'Seaborn', 'OpenCV', 'VS Code', 'PyCharm', 'Visual Studio', 'YOLOv9', 'LLMs', 'LLaMa', 'LangChain', 'Neo4J', 'NLP', 'OCR'], 'domain_knowledge': ['Machine Learning', 'Deep Learning', 'Computer Vision', 'Generative AI', 'Data Mining', 'Data Analysis', 'Database Management', 'Operating Systems', 'Computer Networks', 'Statistics']}


In [79]:
def get_all_node_names():
    query = "MATCH (n) RETURN n"
    nodes = graph.query(query)
    for node in nodes:
        print(node)
    return [node['n']['id'] for node in nodes]

node_names = get_all_node_names()
print("Node Names:")
for name in node_names:
    print(name)


{'n': {'id': 'Api Design'}}
{'n': {'id': 'System Design'}}
{'n': {'id': 'Cloud Architecture'}}
{'n': {'id': 'Mathematics'}}
{'n': {'id': 'Computer Science'}}
{'n': {'id': 'Programming Paradigm'}}
{'n': {'id': 'Programming Language'}}
{'n': {'id': 'Programming Concept'}}
{'n': {'id': 'Database Language'}}
{'n': {'id': 'Database Type'}}
{'n': {'id': 'Database Management System'}}
{'n': {'id': 'Database Design'}}
{'n': {'id': 'Database Systems'}}
{'n': {'id': 'Artificial Intelligence'}}
{'n': {'id': 'Machine Learning'}}
{'n': {'id': 'Deep Learning'}}
{'n': {'id': 'Data Manipulation'}}
{'n': {'id': 'Numerical Computing'}}
{'n': {'id': 'Statistical Visualization'}}
{'n': {'id': 'Nlp'}}
{'n': {'id': 'Sentence Embedding'}}
{'n': {'id': 'Similarity Search'}}
{'n': {'id': 'Parameter-Efficient Fine-Tuning'}}
{'n': {'id': 'Efficient Llm Training'}}
{'n': {'id': 'Supervised Fine-Tuning'}}
{'n': {'id': 'Optimization'}}
{'n': {'id': 'Computer Systems'}}
{'n': {'id': 'Security'}}
{'n': {'id': 'Field'

In [80]:
from langchain_groq import ChatGroq
from typing import Dict, List

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.9,
)

existing_knowledge = node_names

new_skills = resume_skills
relationships = llm.invoke([
    ("system", "You are a knowledge graph relationship specialist. Create clear, ONLY really close relationships between technical concepts."),
    ("human", create_relationship_prompt(new_skills, existing_knowledge))
])

meaning=relationships.content


llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
llm_transformer = LLMGraphTransformer(llm=llm)
text = meaning
documents = [Document(page_content=text)]
graph_documents = llm_transformer.convert_to_graph_documents(documents)
print(f"Nodes:{graph_documents[0].nodes}")
print(f"Relationships:{graph_documents[0].relationships}")

graph = Neo4jGraph(refresh_schema=False)
graph.add_graph_documents(graph_documents)

Nodes:[Node(id='Python', type='Programming language', properties={}), Node(id='Java', type='Programming language', properties={}), Node(id='C/C++', type='Programming language', properties={}), Node(id='Html/Css', type='Web development', properties={}), Node(id='Javascript', type='Programming language', properties={}), Node(id='Tensorflow', type='Machine learning', properties={}), Node(id='Keras', type='Deep learning', properties={}), Node(id='Scikit-Learn', type='Machine learning', properties={}), Node(id='Pandas', type='Data manipulation', properties={}), Node(id='Numpy', type='Numerical computing', properties={}), Node(id='Matplotlib', type='Statistical visualization', properties={}), Node(id='Seaborn', type='Statistical visualization', properties={}), Node(id='Opencv', type='Computer vision', properties={}), Node(id='Vs Code', type='Ide', properties={}), Node(id='Pycharm', type='Ide', properties={}), Node(id='Visual Studio', type='Ide', properties={}), Node(id='Yolov9', type='Comput

In [81]:
from langchain_core.documents import Document
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.9)
llm_transformer = LLMGraphTransformer(llm=llm)
text = meaning
documents = [Document(page_content=text)]
graph_documents = llm_transformer.convert_to_graph_documents(documents)
print(f"Nodes:{graph_documents[0].nodes}")
print(f"Relationships:{graph_documents[0].relationships}")

Nodes:[Node(id='Python', type='Programming language', properties={}), Node(id='Java', type='Programming language', properties={}), Node(id='C/C++', type='Programming language', properties={}), Node(id='Html/Css', type='Web development', properties={}), Node(id='Javascript', type='Programming language', properties={}), Node(id='Tensorflow', type='Machine learning', properties={}), Node(id='Keras', type='Deep learning', properties={}), Node(id='Scikit-Learn', type='Machine learning', properties={}), Node(id='Pandas', type='Data manipulation', properties={}), Node(id='Numpy', type='Numerical computing', properties={}), Node(id='Matplotlib', type='Statistical visualization', properties={}), Node(id='Seaborn', type='Statistical visualization', properties={}), Node(id='Opencv', type='Computer vision', properties={}), Node(id='Vs Code', type='Ide', properties={}), Node(id='Pycharm', type='Ide', properties={}), Node(id='Visual Studio', type='Ide', properties={}), Node(id='Yolov9', type='Comput

In [82]:
from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(refresh_schema=False)
graph.add_graph_documents(graph_documents)

In [83]:
JD_skills

{'technical_skills': ['SQL',
  'Excel',
  'XML',
  'JavaScript',
  'ETL',
  'SPSS',
  'SAS'],
 'domain_knowledge': ['Data Mining',
  'Statistics',
  'Data Analytics',
  'Machine Learning']}

In [84]:
resume_skills

{'technical_skills': ['Python',
  'SQL',
  'Java',
  'C/C++',
  'HTML/CSS',
  'JavaScript',
  'TensorFlow',
  'Keras',
  'Scikit-learn',
  'Pandas',
  'NumPy',
  'Matplotlib',
  'Seaborn',
  'OpenCV',
  'VS Code',
  'PyCharm',
  'Visual Studio',
  'YOLOv9',
  'LLMs',
  'LLaMa',
  'LangChain',
  'Neo4J',
  'NLP',
  'OCR'],
 'domain_knowledge': ['Machine Learning',
  'Deep Learning',
  'Computer Vision',
  'Generative AI',
  'Data Mining',
  'Data Analysis',
  'Database Management',
  'Operating Systems',
  'Computer Networks',
  'Statistics']}

In [86]:
jd_skil = [skill for skills in JD_skills.values() for skill in skills]
resume_skil = [skill for skills in resume_skills.values() for skill in skills]

In [87]:
resume_skil

['Python',
 'SQL',
 'Java',
 'C/C++',
 'HTML/CSS',
 'JavaScript',
 'TensorFlow',
 'Keras',
 'Scikit-learn',
 'Pandas',
 'NumPy',
 'Matplotlib',
 'Seaborn',
 'OpenCV',
 'VS Code',
 'PyCharm',
 'Visual Studio',
 'YOLOv9',
 'LLMs',
 'LLaMa',
 'LangChain',
 'Neo4J',
 'NLP',
 'OCR',
 'Machine Learning',
 'Deep Learning',
 'Computer Vision',
 'Generative AI',
 'Data Mining',
 'Data Analysis',
 'Database Management',
 'Operating Systems',
 'Computer Networks',
 'Statistics']

# **Calculation of Resume and JD Matching Score**

In [88]:
from typing import List, Dict, Tuple

def calculate_skill_match_score(jd_skills: List[str], resume_skills: List[str], graph) -> Tuple[float, Dict]:
    """
    Calculate the skill match score between a job description and resume.
    Returns both the final score and detailed metrics.
    
    The scoring considers:
    1. Path distances between matched skills
    2. Coverage ratio (how many JD skills were matched)
    3. Relevance ratio (quality of the matches based on distance)
    """
    print("Testing JD skills...")
    jd_skill_ids = get_skill_ids(jd_skills, graph)
    print("\nTesting Resume skills...")
    resume_skill_ids = get_skill_ids(resume_skills, graph)

    query = """
    UNWIND $jd_skill_ids AS jd_skill_id
    UNWIND $resume_skill_ids AS resume_skill_id
    MATCH (jd_skill) WHERE id(jd_skill) = jd_skill_id
    MATCH (resume_skill) WHERE id(resume_skill) = resume_skill_id
    MATCH path = shortestPath((resume_skill)-[*..10]-(jd_skill))
    WHERE id(jd_skill) <> id(resume_skill)  // Exclude self-referential paths
    RETURN 
        id(jd_skill) AS jd_skill,
        id(resume_skill) AS resume_skill,
        length(path) AS distance
    """

    params = {'jd_skill_ids': jd_skill_ids, 'resume_skill_ids': resume_skill_ids}
    try:
        results = graph.query(query, params)
    except Exception as e:
        print(f"Debug: Query execution failed with error: {e}")
        return 0.0, {"error": str(e)}

    total_distance_score = 0
    matched_jd_skills = set()
    distance_scores = []  
    
    for result in results:
        distance = result.get('distance')
        if distance is None:
            continue
        
        distance_score = 1.0 / (distance + 1)
        total_distance_score += distance_score
        distance_scores.append(distance_score)
        
        matched_jd_skills.add(result['jd_skill'])

    total_jd_skills = len(jd_skill_ids)
    matched_skills_count = len(matched_jd_skills)
    coverage_ratio = matched_skills_count / total_jd_skills if total_jd_skills > 0 else 0

    avg_distance_score = (sum(distance_scores) / len(distance_scores)) if distance_scores else 0
    
    coverage_weight = 0.6
    relevance_weight = 1 - coverage_weight
    
    final_score = (coverage_weight * coverage_ratio) + (relevance_weight * avg_distance_score)

    metrics = {
        "final_score": final_score,
        "coverage_ratio": coverage_ratio,
        "avg_distance_score": avg_distance_score,
        "matched_skills_count": matched_skills_count,
        "total_jd_skills": total_jd_skills,
        "unmatched_skills_count": total_jd_skills - matched_skills_count
    }

    print(f"\nDetailed Matching Metrics:")
    print(f"Coverage Ratio: {coverage_ratio:.2f}")
    print(f"Average Distance Score: {avg_distance_score:.2f}")
    print(f"Matched Skills: {matched_skills_count}/{total_jd_skills}")
    print(f"Final Score: {final_score:.2f}")

    return final_score, metrics

def test_single_user_and_jd(graph, jd_skills: Dict[str, List[str]], resume_skills: Dict[str, List[str]]):
    """
    Test matching with sample skills and print detailed metrics.
    """
    jd_skills_flat = normalize_skills([skill for skills in jd_skills.values() for skill in skills])
    resume_skills_flat = normalize_skills([skill for skills in resume_skills.values() for skill in skills])
    
    match_score, metrics = calculate_skill_match_score(jd_skills_flat, resume_skills_flat, graph)
    
    print("\nMatch Analysis Summary:")
    print(f"Final Match Score: {match_score:.2f}")
    print(f"Coverage of JD Skills: {metrics['coverage_ratio']*100:.1f}%")
    print(f"Average Quality of Matches: {metrics['avg_distance_score']:.2f}")
    print(f"Unmatched JD Skills: {metrics['unmatched_skills_count']}")

def normalize_skills(skills: List[str]) -> List[str]:
    return [skill.lower() for skill in skills]

def get_skill_ids(skills: List[str], graph) -> List[int]:
    skill_ids = []
    query = """
    MATCH (n) 
    WHERE toLower(n.id) = $skill_name
    RETURN id(n) AS node_id, n.id as found_name
    """
    
    for skill in skills:
        try:
            result = graph.query(query, {'skill_name': skill})
            if result:
                skill_ids.append(result[0]['node_id'])
                print(f"Found skill: {result[0]['found_name']}")
            else:
                print(f"No match found for: {skill}")
        except Exception as e:
            print(f"Query error for {skill}: {e}")
    
    return skill_ids


test_single_user_and_jd(graph, JD_skills, resume_skills)

Testing JD skills...
Found skill: Sql
Found skill: Excel
Found skill: Xml
Found skill: Javascript
Found skill: Etl
Found skill: Spss
Found skill: Sas
Found skill: Data Mining
Found skill: Statistics
Found skill: Data Analytics
Found skill: Machine Learning

Testing Resume skills...
Found skill: Python
Found skill: Sql
Found skill: Java
Found skill: C/C++
Found skill: Html/Css
Found skill: Javascript
Found skill: Tensorflow
Found skill: Keras
Found skill: Scikit-Learn
Found skill: Pandas
Found skill: Numpy
Found skill: Matplotlib
Found skill: Seaborn
Found skill: Opencv
Found skill: Vs Code
Found skill: Pycharm
Found skill: Visual Studio
Found skill: Yolov9
Found skill: Llms
Found skill: Llama
Found skill: Langchain
Found skill: Neo4J
Found skill: Nlp
Found skill: Ocr
Found skill: Machine Learning
Found skill: Deep Learning
Found skill: Computer Vision
Found skill: Generative Ai
Found skill: Data Mining
Found skill: Data Analysis
Found skill: Database Management
Found skill: Operating S

# **Calculation of Multiple Users Resume Scores**

In [89]:
import pandas as pd
from pathlib import Path
from typing import List, Dict, Tuple
import json
import csv


def normalize_skills(skills: List[str]) -> List[str]:
    return [skill.lower() for skill in skills]

def get_skill_ids(skills: List[str], graph) -> List[int]:
    skill_ids = []
    query = """
    MATCH (n) 
    WHERE toLower(n.id) = $skill_name
    RETURN id(n) AS node_id, n.id as found_name
    """
    
    for skill in skills:
        try:
            result = graph.query(query, {'skill_name': skill})
            if result:
                skill_ids.append(result[0]['node_id'])
                print(f"Found skill: {result[0]['found_name']}")
            else:
                print(f"No match found for: {skill}")
        except Exception as e:
            print(f"Query error for {skill}: {e}")
    
    return skill_ids

def calculate_skill_match_score(jd_skills: List[str], resume_skills: List[str], graph) -> Tuple[float, Dict]:
    print("Testing JD skills...")
    jd_skill_ids = get_skill_ids(jd_skills, graph)
    print("\nTesting Resume skills...")
    resume_skill_ids = get_skill_ids(resume_skills, graph)

    query = """
    UNWIND $jd_skill_ids AS jd_skill_id
    UNWIND $resume_skill_ids AS resume_skill_id
    MATCH (jd_skill) WHERE id(jd_skill) = jd_skill_id
    MATCH (resume_skill) WHERE id(resume_skill) = resume_skill_id
    MATCH path = shortestPath((resume_skill)-[*..10]-(jd_skill))
    WHERE id(jd_skill) <> id(resume_skill)
    RETURN 
        id(jd_skill) AS jd_skill,
        id(resume_skill) AS resume_skill,
        length(path) AS distance
    """

    params = {'jd_skill_ids': jd_skill_ids, 'resume_skill_ids': resume_skill_ids}
    try:
        results = graph.query(query, params)
    except Exception as e:
        print(f"Debug: Query execution failed with error: {e}")
        return 0.0, {"error": str(e)}

    total_distance_score = 0
    matched_jd_skills = set()
    distance_scores = []  
    
    for result in results:
        distance = result.get('distance')
        if distance is None:
            continue
        
        distance_score = 1.0 / (distance + 1)
        total_distance_score += distance_score
        distance_scores.append(distance_score)
        matched_jd_skills.add(result['jd_skill'])

    total_jd_skills = len(jd_skill_ids)
    matched_skills_count = len(matched_jd_skills)
    coverage_ratio = matched_skills_count / total_jd_skills if total_jd_skills > 0 else 0
    avg_distance_score = (sum(distance_scores) / len(distance_scores)) if distance_scores else 0
    
    coverage_weight = 0.6
    relevance_weight = 1 - coverage_weight
    final_score = (coverage_weight * coverage_ratio) + (relevance_weight * avg_distance_score)

    metrics = {
        "final_score": final_score,
        "coverage_ratio": coverage_ratio,
        "avg_distance_score": avg_distance_score,
        "matched_skills_count": matched_skills_count,
        "total_jd_skills": total_jd_skills,
        "unmatched_skills_count": total_jd_skills - matched_skills_count
    }

    print(f"\nDetailed Matching Metrics:")
    print(f"Coverage Ratio: {coverage_ratio:.2f}")
    print(f"Average Distance Score: {avg_distance_score:.2f}")
    print(f"Matched Skills: {matched_skills_count}/{total_jd_skills}")
    print(f"Final Score: {final_score:.2f}")

    return final_score, metrics


def create_dataframe_safe(results_list):
    """Try multiple methods to create DataFrame."""
    if not results_list:
        return None
    
    # Method 1: from_records
    try:
        print("\nAttempting DataFrame creation with from_records...")
        df = pd.DataFrame.from_records(results_list)
        print("✓ Success with from_records!")
        return df
    except:
        pass
    
    # Method 2: from_dict
    try:
        print("Attempting DataFrame creation with from_dict...")
        df = pd.DataFrame.from_dict({i: row for i, row in enumerate(results_list)}, orient='index')
        print("✓ Success with from_dict!")
        return df
    except:
        pass
    
    # Method 3: Column by column
    try:
        print("Attempting DataFrame creation column by column...")
        columns = list(results_list[0].keys())
        data = {col: [row[col] for row in results_list] for col in columns}
        df = pd.DataFrame(data)
        print("✓ Success with column-by-column!")
        return df
    except:
        pass
    
    # Method 4: Via CSV
    try:
        print("Attempting DataFrame creation via CSV...")
        import tempfile
        with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.csv', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=results_list[0].keys())
            writer.writeheader()
            writer.writerows(results_list)
            temp_path = f.name
        
        df = pd.read_csv(temp_path)
        import os
        os.unlink(temp_path)
        print("✓ Success via CSV!")
        return df
    except:
        pass
    
    print("✗ All DataFrame creation methods failed - using fallback display")
    return None

def process_multiple_resumes(graph, jd_skills: Dict[str, List[str]], resume_folder_path: str):
    """Process multiple resumes with robust error handling."""
    jd_skills_flat = normalize_skills([skill for skills in jd_skills.values() for skill in skills])
    results_list = []
    
    resume_folder = Path(resume_folder_path)
    
    if not resume_folder.exists():
        print(f"Error: Folder {resume_folder_path} does not exist!")
        return None, results_list
    
    resume_files = list(resume_folder.glob("*.pdf"))
    
    if not resume_files:
        print(f"No PDF files found in {resume_folder_path}")
        return None, results_list
    
    print(f"\nFound {len(resume_files)} resume(s) to process\n")
    print("="*80)
    
    for idx, resume_file in enumerate(resume_files, 1):
        print(f"\n[{idx}/{len(resume_files)}] Processing: {resume_file.name}")
        print("-"*80)
        
        try:
            resume_text = extract_resume_text(str(resume_file))
            resume_skills_dict = extract_skills(resume_text)
            
            resume_skills_flat = normalize_skills([
                skill for skills in resume_skills_dict.values() for skill in skills
            ])
            
            print(f"Extracted {len(resume_skills_flat)} skills from resume")
            
            match_score, metrics = calculate_skill_match_score(
                jd_skills_flat, 
                resume_skills_flat, 
                graph
            )
            
            results_list.append({
                'Resume_Name': resume_file.name,
                'Match_Score': round(match_score, 4),
                'Coverage_Ratio': round(metrics['coverage_ratio'], 4),
                'Avg_Distance_Score': round(metrics['avg_distance_score'], 4),
                'Matched_Skills': metrics['matched_skills_count'],
                'Total_JD_Skills': metrics['total_jd_skills'],
                'Unmatched_Skills': metrics['unmatched_skills_count'],
                'Total_Resume_Skills': len(resume_skills_flat)
            })
            
            print(f"✓ Successfully processed {resume_file.name}")
            
        except Exception as e:
            print(f"✗ Error processing {resume_file.name}: {str(e)}")
            results_list.append({
                'Resume_Name': resume_file.name,
                'Match_Score': 0.0,
                'Coverage_Ratio': 0.0,
                'Avg_Distance_Score': 0.0,
                'Matched_Skills': 0,
                'Total_JD_Skills': len(jd_skills_flat),
                'Unmatched_Skills': len(jd_skills_flat),
                'Total_Resume_Skills': 0
            })
    
    print("\n" + "="*80)
    print("\nProcessing Complete!")
    
    # Sort results
    results_list.sort(key=lambda x: x['Match_Score'], reverse=True)
    
    # Try to create DataFrame
    results_df = create_dataframe_safe(results_list)
    
    return results_df, results_list



results_df, results_list = process_multiple_resumes(
    graph=graph,
    jd_skills=JD_skills,
    resume_folder_path='/kaggle/input/multiple-resumes/resumes'
)

print("\n" + "="*80)
print("FINAL RESULTS - RANKED BY MATCH SCORE")
print("="*80)

if results_df is not None:
    # DataFrame created successfully
    print(results_df.to_string(index=False))
    
    print("\n" + "="*80)
    print("TOP 5 CANDIDATES")
    print("="*80)
    if len(results_df) >= 5:
        print(results_df.head(5)[['Resume_Name', 'Match_Score', 'Coverage_Ratio', 'Matched_Skills']].to_string(index=False))
    else:
        print(results_df[['Resume_Name', 'Match_Score', 'Coverage_Ratio', 'Matched_Skills']].to_string(index=False))
    
    results_df.to_csv('resume_matching_results.csv', index=False)
    print(f"\n✓ Results saved to resume_matching_results.csv")
    
else:
    # Fallback: Display as formatted table
    print("\n📊 RESULTS TABLE:\n")
    
    header = f"{'Rank':<6} {'Resume Name':<45} {'Score':<8} {'Coverage':<10} {'Matched':<8} {'Total Skills':<12}"
    print(header)
    print("-" * len(header))
    
    for idx, result in enumerate(results_list, 1):
        row = f"{idx:<6} {result['Resume_Name']:<45} {result['Match_Score']:<8.4f} {result['Coverage_Ratio']:<10.2%} {result['Matched_Skills']:<8} {result['Total_Resume_Skills']:<12}"
        print(row)
    
    # Save results manually
    csv_path = 'resume_matching_results.csv'
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=results_list[0].keys())
        writer.writeheader()
        writer.writerows(results_list)
    print(f"\n✓ Results saved to {csv_path}")
    
    json_path = 'resume_matching_results.json'
    with open(json_path, 'w') as f:
        json.dump(results_list, f, indent=2)
    print(f"✓ Results saved to {json_path}")

print("\n" + "="*80)
print("🏆 RANKED CANDIDATES:")
print("="*80)
for idx, result in enumerate(results_list[:5], 1):
    print(f"{idx}. {result['Resume_Name']}: {result['Match_Score']:.4f} ({result['Matched_Skills']}/{result['Total_JD_Skills']} skills matched)")



Found 5 resume(s) to process


[1/5] Processing: Peddamallu_Sai_Gowthami_Resume.pdf
--------------------------------------------------------------------------------
Extracted 13 skills from resume
Testing JD skills...
Found skill: Sql
Found skill: Excel
Found skill: Xml
Found skill: Javascript
Found skill: Etl
Found skill: Spss
Found skill: Sas
Found skill: Data Mining
Found skill: Statistics
Found skill: Data Analytics
Found skill: Machine Learning

Testing Resume skills...
Found skill: Python
Found skill: C
Found skill: C++
Found skill: Sql
Found skill: Java
Found skill: Mysql
Found skill: Windows
Found skill: Linux
Found skill: Lstm
Found skill: Machine Learning
Found skill: Data Structures
Found skill: Algorithms
No match found for: cyber security

Detailed Matching Metrics:
Coverage Ratio: 0.36
Average Distance Score: 0.36
Matched Skills: 4/11
Final Score: 0.36
✓ Successfully processed Peddamallu_Sai_Gowthami_Resume.pdf

[2/5] Processing: BattaVenkataRahul_Resume.pdf
------------